In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Data Preparation

We will use the cleaned Primary (SATEL) and Secondary (Aqar) datasets from Phase 2 to prepare the data for regression modelling.
We will apply regression to both datasets separately to compare their performance.

At first we will load both datasets and check their dimensions.

In [27]:
primary_df = pd.read_csv("cleaned_satel_riyadh_rentals.csv")
secondary_df = pd.read_csv("secondary_cleaned.csv")

print("Primary Dataset:", primary_df.shape)
print("Secondary Dataset:", secondary_df.shape)

Primary Dataset: (218, 13)
Secondary Dataset: (847, 19)


In [28]:
print("Primary Dataset Columns:")
print(primary_df.columns.tolist())

print("\nSecondary Dataset Columns:")
print(secondary_df.columns.tolist())

Primary Dataset Columns:
['property_id', 'neighborhood', 'neighborhood_ar', 'property_type', 'annual_rent_sar', 'area_sqm', 'bedrooms', 'bathrooms', 'furnishing', 'status', 'title_text', 'created_at', 'neighborhood_key']

Secondary Dataset Columns:
['property_id', 'neighborhood_ar', 'neighborhood_key', 'property_type', 'annual_rent_sar', 'area_sqm', 'bedrooms', 'bathrooms', 'living_rooms', 'furnished', 'ac', 'lift', 'maid_room', 'driver_room', 'pool', 'created_at', 'log_annual_rent', 'price_outlier_iqr', 'area_flag']


In [29]:
print("Primary Dataset Missing Values:")
print(primary_df.isnull().sum())

print("\nSecondary Dataset Missing Values:")
print(secondary_df.isnull().sum())

Primary Dataset Missing Values:
property_id          0
neighborhood         0
neighborhood_ar     17
property_type        0
annual_rent_sar      0
area_sqm             1
bedrooms             0
bathrooms            0
furnishing           0
status               0
title_text           0
created_at           0
neighborhood_key     0
dtype: int64

Secondary Dataset Missing Values:
property_id            0
neighborhood_ar        0
neighborhood_key       0
property_type          0
annual_rent_sar        0
area_sqm               0
bedrooms              13
bathrooms            186
living_rooms         201
furnished              0
ac                     0
lift                   0
maid_room              0
driver_room            0
pool                   0
created_at             0
log_annual_rent        0
price_outlier_iqr      0
area_flag              0
dtype: int64


In [30]:
print("Primary Dataset Data Types:")
print(primary_df.dtypes)

print("\nSecondary Dataset Data Types:")
print(secondary_df.dtypes)

Primary Dataset Data Types:
property_id          object
neighborhood         object
neighborhood_ar      object
property_type        object
annual_rent_sar       int64
area_sqm            float64
bedrooms              int64
bathrooms             int64
furnishing           object
status               object
title_text           object
created_at           object
neighborhood_key     object
dtype: object

Secondary Dataset Data Types:
property_id            int64
neighborhood_ar       object
neighborhood_key      object
property_type         object
annual_rent_sar      float64
area_sqm             float64
bedrooms             float64
bathrooms            float64
living_rooms         float64
furnished              int64
ac                     int64
lift                   int64
maid_room              int64
driver_room            int64
pool                   int64
created_at            object
log_annual_rent      float64
price_outlier_iqr       bool
area_flag               bool
dtype: objec

## Feature Selection

We selected five features for regression modelling which are: area, bedrooms, bathrooms, property type, and neighborhood. These features may help explain differences in annual rental prices.

The target variable is annual_rent_sar. We will use the same features for both datasets to make the comparison more consistent.

In [31]:
features = [
    "area_sqm",
    "bedrooms",
    "bathrooms",
    "property_type",
    "neighborhood_key"
]

target = "annual_rent_sar"

print("Selected Features:", features)
print("Target Variable:", target)

Selected Features: ['area_sqm', 'bedrooms', 'bathrooms', 'property_type', 'neighborhood_key']
Target Variable: annual_rent_sar


#### 1-Primary Dataset

In [32]:
X_primary = primary_df[features]
y_primary = primary_df[target]

print("Primary Features:", X_primary.shape)
print("Primary Target:", y_primary.shape)

Primary Features: (218, 5)
Primary Target: (218,)


#### 2-Secondary Dataset

In [33]:
X_secondary = secondary_df[features]
y_secondary = secondary_df[target]

print("Secondary Features:", X_secondary.shape)
print("Secondary Target:", y_secondary.shape)

Secondary Features: (847, 5)
Secondary Target: (847,)


## Train-Test Split

We will split both datasets into 80% training and 20% testing data. The training data will be used to build the regression models, while the testing data will be used to evaluate their performance on unseen data.

#### 1-Primary Dataset

In [34]:
from sklearn.model_selection import train_test_split

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
X_primary,
y_primary,
test_size=0.2,
random_state=42
)
print("Primary Training:", X_train_p.shape)
print("Primary Testing:", X_test_p.shape)

Primary Training: (174, 5)
Primary Testing: (44, 5)


#### 2-Secondary Dataset

In [35]:
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
X_secondary,
y_secondary,
test_size=0.2,
random_state=42
)

print("Secondary Training:", X_train_s.shape)
print("Secondary Testing:", X_test_s.shape)

Secondary Training: (677, 5)
Secondary Testing: (170, 5)


## Handling Missing Values

We will use median imputation to fill the missing numerical values in both datasets. The median will be calculated using only the training data to avoid data leakage. This way, we can keep all records without deleting any rows.

In [36]:
from sklearn.impute import SimpleImputer

numeric_features = ["area_sqm", "bedrooms", "bathrooms"]

numeric_imputer = SimpleImputer(strategy="median")

print("Numerical features:", numeric_features)
print("Imputation method: Median")

Numerical features: ['area_sqm', 'bedrooms', 'bathrooms']
Imputation method: Median


## Encoding Categorical Features

We will use one-hot encoding to convert property type and neighborhood into numerical values so they can be used in regression models. The encoding will be fitted using only the training data to avoid data leakage.

In [37]:
from sklearn.preprocessing import OneHotEncoder

categorical_features = ["property_type", "neighborhood_key"]
categorical_encoder = OneHotEncoder(handle_unknown="ignore")

print("Categorical features:", categorical_features)
print("Encoding method: One-Hot Encoding")

Categorical features: ['property_type', 'neighborhood_key']
Encoding method: One-Hot Encoding


## Preprocessing Pipeline

We will combine median imputation and one-hot encoding into one preprocessing pipeline. This will prepare the numerical and categorical features for regression modelling.

In [38]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
transformers=[
("num", numeric_imputer, numeric_features),
("cat", categorical_encoder, categorical_features)
]
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


## Baseline Regression Model

We will use Linear Regression as our baseline model to predict annual rental prices. The model will be trained separately on the primary and secondary datasets using the same selected features.

#### 1-Primary Dataset

In [39]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.base import clone

primary_model = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("regression", LinearRegression())
])

primary_model.fit(X_train_p, y_train_p)

print("Primary Linear Regression model trained successfully.")

Primary Linear Regression model trained successfully.


#### 2-Secondary Dataset

In [40]:
secondary_model = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("regression", LinearRegression())
])

secondary_model.fit(X_train_s, y_train_s)

print("Secondary Linear Regression model trained successfully.")

Secondary Linear Regression model trained successfully.


## Model Evaluation

We will evaluate both regression models using MAE, RMSE, and R². These metrics will help us understand how well each model predicts annual rental prices using the testing data.

In [41]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Predict rental prices
y_pred_p = primary_model.predict(X_test_p)

# Calculate evaluation metrics
mae_p = mean_absolute_error(y_test_p, y_pred_p)
rmse_p = np.sqrt(mean_squared_error(y_test_p, y_pred_p))
r2_p = r2_score(y_test_p, y_pred_p)

print("Primary Dataset - Linear Regression")
print("MAE:", round(mae_p, 2))
print("RMSE:", round(rmse_p, 2))
print("R²:", round(r2_p, 4))

Primary Dataset - Linear Regression
MAE: 25469.21
RMSE: 34349.62
R²: 0.3226


In [42]:
# Predict rental prices
y_pred_s = secondary_model.predict(X_test_s)

# Calculate evaluation metrics
mae_s = mean_absolute_error(y_test_s, y_pred_s)
rmse_s = np.sqrt(mean_squared_error(y_test_s, y_pred_s))
r2_s = r2_score(y_test_s, y_pred_s)

print("Secondary Dataset - Linear Regression")
print("MAE:", round(mae_s, 2))
print("RMSE:", round(rmse_s, 2))
print("R²:", round(r2_s, 4))

Secondary Dataset - Linear Regression
MAE: 17733.81
RMSE: 33377.37
R²: 0.5188


## Baseline Model Comparison

We will compare the Linear Regression results from both datasets using MAE, RMSE, and R² to understand their prediction performance.

In [43]:
comparison = pd.DataFrame({
"Dataset": ["Primary (SATEL)", "Secondary (Aqar)"],
"MAE": [mae_p, mae_s],
"RMSE": [rmse_p, rmse_s],
"R²": [r2_p, r2_s]
})

comparison.round(2)

,Dataset,MAE,RMSE,R²
0,Primary (SATEL),25469.21,34349.62,0.32
1,Secondary (Aqar),17733.81,33377.37,0.52


## Interpretation

The Linear Regression model performed better on the secondary dataset, with a lower MAE (17,733.81 SAR) and RMSE (33,377.37 SAR) compared to the primary dataset.

The secondary model also achieved a higher R² value of 0.5188, meaning it explains approximately 51.88% of the variation in annual rental prices. In comparison, the primary model explains approximately 32.26%.

Overall, the secondary dataset showed better prediction performance. However, the two datasets have different rental price ranges and distributions, which should be considered when comparing their results.